# 04 · From Jerk to Snap

### Recap & why now
Notebook 03 minimised jerk because it was the tidy next step. For a quadcopter it is one
derivative short, and this notebook follows the chain to find out why.

The argument runs through everything Project 5 built: acceleration is tilt, jerk is
angular rate, and **snap** — the fourth derivative — is angular acceleration, which is
torque, which is the difference between the four motor thrusts. Snap is the derivative
that lands on the actuators.

### Learning objectives
1. Trace the chain from position derivatives down to motor commands.
2. Explain why **snap** is the quantity a quadcopter's motors care about.
3. Build the snap cost matrix and verify it numerically.
4. Derive the minimum-snap septic and compare it with the quintic.
5. Discover that "minimum snap" does not mean what its name suggests.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · The chain

```text
   acceleration  →  tilt angle             (a quadcopter tilts to accelerate)
   jerk          →  rate of tilt change    →  angular velocity
   snap          →  angular acceleration   →  torque  →  motor thrust differences
```

Every step is one of Project 5's results. The last one is the point: a trajectory with
large snap demands large torques, which means large differences between the four motor
thrusts — and that is exactly what runs out when a motor saturates.

Minimising jerk gives a smooth-*looking* path. Minimising snap gives a path that is
gentle on the **actuators**.

In [ ]:
ARM, DRAG = 0.25/np.sqrt(2), 0.016                 # Project 5's geometry: moment arm and drag ratio.
I_xx, m = 0.01, 1.0                                # Inertia about roll, and mass.

print("  snap [m/s^4]   angular accel   torque needed   motor imbalance")
for snap in (1.0, 5.0, 20.0, 50.0):
    ang_acc = snap/g                               # d(tilt)/dt^2 ~ snap/g for small tilts.
    torque = I_xx*ang_acc                          # Euler's equation, from Project 5.
    imbalance = torque/(2*ARM)                     # The thrust difference that produces it.
    print("  %12.1f %15.2f rad/s^2 %13.4f N m %13.4f N" % (snap, ang_acc, torque, imbalance))

print("\nA hover thrust on this vehicle is %.2f N per motor, so the last row asks for a %.0f%%" %
      (m*g/4, 100*(50.0/g*I_xx/(2*ARM))/(m*g/4)))
print("imbalance between motors. Snap is not an abstract smoothness measure — it is a")
print("direct claim about how far apart the four motor commands have to be.")

## 2 · The snap cost

Same machinery as Notebook 03, one derivative deeper:

$$J = \int_0^T \left(p^{(4)}(t)\right)^2 dt = c^\top Q\, c,
\qquad Q_{ij} = \frac{[i]_4\,[j]_4}{i+j-7}\, T^{\,i+j-7}$$

for $i, j \ge 4$, where $[i]_4 = i(i-1)(i-2)(i-3)$, and zero otherwise. The first
**four** coefficients are free now, because $c_0$ through $c_3$ contribute no snap at all.

In [ ]:
n, T = 8, 2.5
c_test = np.array([0.3, -1.2, 0.4, 2.0, -0.7, 1.1, -0.2, 0.05])
Q4 = cost_matrix(n, T, der=4)

analytic = c_test @ Q4 @ c_test
grid = np.linspace(0, T, 200001)
numeric = np.trapezoid([poly_val(c_test, t_, 4)**2 for t_ in grid], grid)
print("snap cost from the matrix      : %.6f" % analytic)
print("snap cost by numerical integral: %.6f  (relative difference %.1e ✔)" %
      (numeric, abs(analytic-numeric)/numeric))

print("\nthe top-left corner of Q, rounded:")
print(np.round(Q4[:6, :6], 2))
print("\nFour empty rows and columns now, against three for jerk. Each derivative you go up")
print("frees one more coefficient — and demands one more boundary condition to pin it down.")

## 3 · The minimum-snap septic

Eight conditions — position, velocity, acceleration **and jerk** at both ends — pin down
eight coefficients, so once again the answer is forced:

$$s(\tau) = 35\tau^4 - 84\tau^5 + 70\tau^6 - 20\tau^7$$

The extra pair of conditions is the interesting part. Zero jerk at the endpoints means
the trajectory asks for zero *angular rate* there — which is what a drone sitting still
at a waypoint actually has.

In [ ]:
def solve_exact(conditions, n):
    A = np.array([row for row, _ in conditions]); b = np.array([val for _, val in conditions])
    return np.linalg.solve(A, b)

T = 1.0
eight = [(deriv_row(8, 0.0, d), 0.0) for d in range(4)] + \
        [(deriv_row(8, T, d), 1.0 if d == 0 else 0.0) for d in range(4)]
c_snap = solve_exact(eight, 8)
c_jerk = np.concatenate([solve_exact([(deriv_row(6, 0.0, d), 0.0) for d in range(3)] +
                                     [(deriv_row(6, T, d), 1.0 if d == 0 else 0.0) for d in range(3)], 6),
                         [0.0, 0.0]])              # Pad the quintic so both live in the same space.
print("minimum snap:", np.round(c_snap, 4), " = 35t^4 - 84t^5 + 70t^6 - 20t^7 ✔")

tau = np.linspace(0, 1, 400)
names = ["position", "velocity", "acceleration", "jerk", "snap"]
fig, axes = plt.subplots(1, 5, figsize=(16, 2.6))
for d, ax_ in enumerate(axes):
    ax_.plot(tau, [poly_val(c_jerk, t_, d) for t_ in tau], color="C0", lw=2, label="min jerk")
    ax_.plot(tau, [poly_val(c_snap, t_, d) for t_ in tau], color="C3", lw=2, label="min snap")
    ax_.set_title(names[d], fontsize=10); ax_.set_xlabel(r"$\tau$")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("The position curves are nearly identical, which is why min-jerk tracked well in Project 5.")
print("The real difference is at the ENDS of the jerk plot: min-jerk starts with a jerk")
print("discontinuity of %.0f, which the drone feels as a sudden demand for angular rate." %
      abs(poly_val(c_jerk, 0.0, 3)))
print("Min-snap eases into it at %.0f, and pays for that with a higher peak in the middle." %
      abs(poly_val(c_snap, 0.0, 3)))

## 4 · A result that contradicts the name

Now compute the snap cost of both curves. The answer is not the one the naming suggests,
and understanding why is worth more than the formula.

In [ ]:
Q4 = cost_matrix(8, 1.0, der=4)
print("  curve                                   snap cost")
print("  minimum-JERK quintic %27.1f" % (c_jerk @ Q4 @ c_jerk))
print("  standard minimum-SNAP septic %19.1f" % (c_snap @ Q4 @ c_snap))
print("  -> the min-SNAP curve is WORSE on snap. That is not a bug.\n")

def solve_min(conditions, n, Q):
    """Minimise c^T Q c subject to the given conditions, via the KKT system."""
    A = np.array([r for r, _ in conditions]); b = np.array([v for _, v in conditions])
    KKT = np.block([[2*Q, A.T], [A, np.zeros((len(b), len(b)))]])
    return np.linalg.solve(KKT, np.concatenate([np.zeros(n), b]))[:n]

six = [(deriv_row(8, 0.0, d), 0.0) for d in range(3)] + \
      [(deriv_row(8, 1.0, d), 1.0 if d == 0 else 0.0) for d in range(3)]
c_free = solve_min(six, 8, Q4)                     # Jerk left FREE at the endpoints.
c_zero = solve_min(six + [(deriv_row(8, 0.0, 3), 0.0), (deriv_row(8, 1.0, 3), 0.0)], 8, Q4)

print("  minimising snap, jerk FREE at the ends %10.1f  coeffs %s" %
      (c_free @ Q4 @ c_free, np.round(c_free, 2)))
print("  minimising snap, jerk ZERO at the ends %10.1f  coeffs %s" %
      (c_zero @ Q4 @ c_zero, np.round(c_zero, 2)))
print("  the second matches the textbook septic to %.1e ✔" % np.abs(c_zero - c_snap).max())

print("\nSo the ranking on snap is: jerk-free < min-jerk quintic < 'minimum snap'.")
print("'Minimum snap' does not mean the lowest snap of any trajectory. It means the lowest snap")
print("among trajectories that ALSO start and end with zero jerk — and every constraint you add")
print("can only raise the achievable minimum. What those two extra conditions buy is a clean")
print("handover at the endpoints: zero jerk means zero angular-rate demand where the drone is")
print("standing still, so it never has to snap into a tilt the instant it starts moving.")

## 🧪 Try it yourself

**E1.** Constraints can only raise a minimum, never lower it. Why is that true in
general, and what does it mean when comparing two optimisation results?

**E2.** Build the cost matrix for the fifth derivative (crackle) and see whether the
pattern of empty rows continues. How many boundary conditions would that need?

In [ ]:
# --- Solution E1 ---
print("E1: because the feasible set SHRINKS. Adding a constraint removes candidates, and the best")
print("    of a smaller set can never beat the best of a larger one containing it. So a lower cost")
print("    means one of two things: a genuinely better solution, or a laxer specification.")
print("    When comparing two optimisation results, always check they solved the same problem —")
print("    Notebook 03's cubic and this notebook's septic both looked 'better' on a cost, and in")
print("    both cases the reason was fewer constraints rather than more skill.")

# --- Solution E2 ---
print("\nE2:  derivative   empty rows in Q   coefficients needed   polynomial order")
for der in (2, 3, 4, 5):
    Q_ = cost_matrix(12, 1.0, der=der)
    empty = sum(1 for i in range(12) if np.abs(Q_[i]).max() < 1e-12)
    n_needed = 2*(der + 1)                         # Conditions up to the (der-1)th at both ends... plus one.
    print("    %10d %17d %21d %18d" % (der, empty, n_needed, n_needed - 1))
print("    The pattern holds: penalising the d-th derivative leaves the first d coefficients free,")
print("    and pinning them down needs conditions up to the d-th derivative at both ends.")
print("    Crackle would need a 9th-order polynomial and jerk-plus-snap conditions at each end —")
print("    which is what you do for a vehicle whose ACTUATORS have their own dynamics, since then")
print("    even torque cannot change instantly.")

## 🚁 Mini-project: the demand each curve places on the motors

Animate the two curves together, showing the tilt they demand and the motor imbalance
that implies. The position traces are nearly identical; the motor traces are not.

In [ ]:
T, dist = 2.0, 3.0
times = np.linspace(0, T, 150)
def profile(cc, t_):
    tau = t_/T
    return np.array([dist*poly_val(cc, tau, d)/T**d for d in range(5)])
tr_jerk = np.array([profile(c_jerk, t_) for t_ in times])
tr_snap = np.array([profile(c_snap, t_) for t_ in times])

fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 4.8), gridspec_kw={"height_ratios": [1, 1]})

def frame(k):
    a1.clear(); a2.clear()
    for tr, name, col in [(tr_jerk, "min jerk", "C0"), (tr_snap, "min snap", "C3")]:
        a1.plot(times[:k+1], np.degrees(np.arctan(tr[:k+1, 2]/g)), color=col, lw=1.8, label=name)
        imbalance = I_xx*(tr[:k+1, 4]/g)/(2*ARM)   # Snap -> angular accel -> torque -> imbalance.
        a2.plot(times[:k+1], imbalance, color=col, lw=1.8)
    a1.set_xlim(0, T); a1.set_ylim(-25, 25); a1.legend(fontsize=8)
    a1.set_ylabel("tilt demanded [deg]"); a1.set_title("t = %4.2f s" % times[k], fontsize=10)
    a2.set_xlim(0, T); a2.set_ylim(-0.06, 0.06)
    a2.set_xlabel("time [s]"); a2.set_ylabel("motor imbalance [N]")
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(times), interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Mellinger and Kumar showed in 2011 that a quadrotor is
> *differentially flat* in position and yaw: given a smooth enough position trajectory,
> every other state — attitude, angular rate, and each of the four motor commands — can be
> computed directly from that trajectory and its derivatives. Snap appears in that
> computation because motor thrust differences do. Minimising snap is therefore not an
> aesthetic choice; it is minimising, as directly as a polynomial can, the thing the
> motors have to deliver.

**Where next.** So far every problem has had exactly as many conditions as coefficients,
so nothing was ever optimised. Notebook 05 changes that: more coefficients than
conditions, and a cost matrix that finally has a job.